# Text Sentiment Classification by Custom LSTM
## Homework 4 - Improved Baseline

This notebook implements a custom LSTM model for text sentiment classification without using external packages like gensim, transformers, or nltk.

**Important**: Before running, make sure to:
1. Mount your Google Drive with the data files in '6001-HW4' folder
2. Enable GPU: Runtime → Change runtime type → Hardware accelerator → GPU

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# Import required packages (only torch, numpy, pandas allowed)
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import re
import os
from collections import Counter
import random
from torch.utils import data

# Mount Google Drive and set path prefix
from google.colab import drive
drive.mount('/content/drive')

# Set path to your HW4 folder in Google Drive
path_prefix = '/content/drive/MyDrive/6001-HW4'

# Tokenization pattern
pattern = r'\w+|[^\w\s]'

## Data Loading Functions

In [ ]:
def load_training_data(path='train.csv'):
    """Load training data from CSV file."""
    data = pd.read_csv(path).to_numpy()
    lines = data[:,0]
    x = [re.findall(pattern, line) for line in lines]
    if 'unlabel' not in path:
        y = data[:,1]
        return x, y
    else:
        return x

def load_testing_data(path='test.csv'):
    """Load testing data from CSV file."""
    data = pd.read_csv(path).to_numpy()
    lines = data[:,1]
    x = [re.findall(pattern, line) for line in lines]
    return x

def evaluation(outputs, labels):
    """Calculate accuracy."""
    outputs[outputs>=0.5] = 1
    outputs[outputs<0.5] = 0
    correct = torch.sum(torch.eq(outputs, labels)).item()
    return correct

## Custom Word Embedding (No gensim!)

We implement a simple vocabulary-based embedding approach using randomized embeddings initialized from a normal distribution.

In [ ]:
class SimpleVocabEmbedding:
    """
    A simple vocabulary-based embedding approach without external packages.
    Uses randomized embeddings initialized from a normal distribution.
    """
    def __init__(self, sentences, vector_size=250, min_count=2):
        self.vector_size = vector_size
        self.min_count = min_count
        self.wv = {}
        self._build_vocab(sentences)
    
    def _count_words(self, sentences):
        word_counts = Counter()
        for sentence in sentences:
            word_counts.update(sentence)
        return word_counts
    
    def _build_vocab(self, sentences):
        print("Building vocabulary...")
        word_counts = self._count_words(sentences)
        
        # Filter by min_count
        vocab = [word for word, count in word_counts.items() if count >= self.min_count]
        print(f"Vocabulary size: {len(vocab)} (min_count={self.min_count})")
        
        # Initialize embeddings with small random values
        for word in vocab:
            self.wv[word] = np.random.randn(self.vector_size).astype(np.float32) * 0.1
        
        print("Embeddings initialized.")
    
    def save(self, path):
        torch.save(self.wv, path)
    
    @classmethod
    def load(cls, path):
        model = cls.__new__(cls)
        model.wv = torch.load(path, weights_only=False)
        return model

## Data Preprocessing

In [ ]:
class Preprocess():
    def __init__(self, sentences, sen_len, w2v_path="./w2v.model"):
        self.w2v_path = w2v_path
        self.sentences = sentences
        self.sen_len = sen_len
        self.idx2word = []
        self.word2idx = {}
        self.embedding_matrix = []
    
    def get_w2v_model(self):
        self.wv_dict = torch.load(self.w2v_path, weights_only=False)
        self.embedding_dim = len(list(self.wv_dict.values())[0])
    
    def add_embedding(self, word):
        vector = torch.empty(1, self.embedding_dim)
        torch.nn.init.uniform_(vector)
        self.word2idx[word] = len(self.word2idx)
        self.idx2word.append(word)
        self.embedding_matrix = torch.cat([self.embedding_matrix, vector], 0)
    
    def make_embedding(self, load=True):
        print("Get embedding ...")
        if load:
            print("loading word to vec model ...")
            self.get_w2v_model()
        else:
            raise NotImplementedError

        for i, word in enumerate(self.wv_dict):
            print('get words #{}'.format(i+1), end='\r')
            self.word2idx[word] = len(self.word2idx)
            self.idx2word.append(word)
            self.embedding_matrix.append(torch.tensor(self.wv_dict[word], dtype=torch.float32))
        print('')
        self.embedding_matrix = torch.stack(self.embedding_matrix)
        self.add_embedding("<PAD>")
        self.add_embedding("<UNK>")
        print("total words: {}".format(len(self.embedding_matrix)))
        return self.embedding_matrix
    
    def pad_sequence(self, sentence):
        if len(sentence) > self.sen_len:
            sentence = sentence[:self.sen_len]
        else:
            pad_len = self.sen_len - len(sentence)
            for _ in range(pad_len):
                sentence.append(self.word2idx["<PAD>"])
        assert len(sentence) == self.sen_len
        return sentence
    
    def sentence_word2idx(self):
        sentence_list = []
        for i, sen in enumerate(self.sentences):
            print('sentence count #{}'.format(i+1), end='\r')
            sentence_idx = []
            for word in sen:
                if word in self.word2idx:
                    sentence_idx.append(self.word2idx[word])
                else:
                    sentence_idx.append(self.word2idx["<UNK>"])
            sentence_idx = self.pad_sequence(sentence_idx)
            sentence_list.append(sentence_idx)
        return torch.LongTensor(sentence_list)
    
    def labels_to_tensor(self, y):
        y = [int(label) for label in y]
        return torch.LongTensor(y)

## Dataset Class

In [ ]:
class SenDataset(data.Dataset):
    def __init__(self, X, y):
        self.data = X
        self.label = y
    def __getitem__(self, idx):
        if self.label is None: 
            return self.data[idx]
        return self.data[idx], self.label[idx]
    def __len__(self):
        return len(self.data)

## Custom LSTM Model

This is our custom LSTM-like model with the following features:
- Bidirectional LSTM for better context understanding
- Multiple fully connected layers with dropout for regularization
- Tunable hyperparameters

In [ ]:
class LSTM_Net(nn.Module):
    def __init__(self, embedding, embedding_dim, hidden_dim, num_layers, dropout=0.5, fix_embedding=True, bidirectional=False):
        super(LSTM_Net, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(embedding.size(0), embedding.size(1))
        self.embedding.weight = nn.Parameter(embedding)
        self.embedding.weight.requires_grad = False if fix_embedding else True
        self.embedding_dim = embedding.size(1)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        # Classifier with multiple layers
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_output_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, inputs):
        embedded = self.embedding(inputs)
        lstm_out, (hidden, cell) = self.lstm(embedded, None)
        
        # Use last hidden state from both directions if bidirectional
        if self.bidirectional:
            # Concatenate the final forward and backward hidden states
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]
        
        output = self.classifier(hidden)
        return output

## Training Function

In [ ]:
def training(batch_size, n_epoch, lr, model_dir, train, valid, model, device):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('\nstart training, parameter total:{}, trainable:{}\n'.format(total, trainable))
    
    model.train()
    criterion = nn.BCELoss()
    t_batch = len(train)
    v_batch = len(valid)
    
    # Adam optimizer with weight decay
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)
    
    total_loss, total_acc, best_acc = 0, 0, 0
    
    for epoch in range(n_epoch):
        model.train()
        total_loss, total_acc = 0, 0
        
        for i, (inputs, labels) in enumerate(train):
            inputs = inputs.to(device, dtype=torch.long)
            labels = labels.to(device, dtype=torch.float)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            outputs = outputs.squeeze()
            
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            correct = evaluation(outputs.detach(), labels)
            total_acc += correct
            total_loss += loss.item()
            
            if (i + 1) % 50 == 0:
                print('[ Epoch{}: {}/{} ] loss:{:.3f} acc:{:.3f} '.format(
                    epoch+1, i+1, t_batch, loss.item(), correct*100/batch_size), end='\r')
        
        train_acc = total_acc / (t_batch * batch_size)
        print('\nTrain | Loss:{:.5f} Acc: {:.3f}'.format(total_loss/t_batch, train_acc*100))
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_loss, val_acc = 0, 0
            for i, (inputs, labels) in enumerate(valid):
                inputs = inputs.to(device, dtype=torch.long)
                labels = labels.to(device, dtype=torch.float)
                
                outputs = model(inputs)
                outputs = outputs.squeeze()
                
                loss = criterion(outputs, labels)
                correct = evaluation(outputs, labels)
                val_acc += correct
                val_loss += loss.item()
            
            val_acc = val_acc / (v_batch * batch_size)
            print("Valid | Loss:{:.5f} Acc: {:.3f} ".format(val_loss/v_batch, val_acc*100))
            
            scheduler.step(val_acc)
            
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(model, "{}/ckpt.model".format(model_dir))
                print('saving model with acc {:.3f}'.format(val_acc*100))
        
        print('-----------------------------------------------')
    
    return best_acc

## Testing Function

In [ ]:
def testing(batch_size, test_loader, model, device):
    model.eval()
    ret_output = []
    with torch.no_grad():
        for i, inputs in enumerate(test_loader):
            inputs = inputs.to(device, dtype=torch.long)
            outputs = model(inputs)
            outputs = outputs.squeeze()
            outputs[outputs>=0.5] = 1
            outputs[outputs<0.5] = 0
            ret_output += outputs.int().tolist()
    
    return ret_output

## Self-Training with Unlabeled Data

In [ ]:
def self_training_iteration(model, unlabeled_loader, device, threshold=0.95):
    """Generate pseudo-labels for unlabeled data"""
    model.eval()
    pseudo_labeled_data = []
    
    with torch.no_grad():
        for inputs in unlabeled_loader:
            inputs = inputs.to(device, dtype=torch.long)
            outputs = model(inputs)
            outputs = outputs.squeeze()
            
            # Only keep high-confidence predictions
            confident_mask = ((outputs > threshold) | (outputs < (1 - threshold)))
            
            for i in range(len(inputs)):
                if confident_mask[i]:
                    pseudo_label = int(outputs[i] >= 0.5)
                    pseudo_labeled_data.append((inputs[i].cpu(), pseudo_label))
    
    return pseudo_labeled_data

## Main Training Pipeline

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
train_with_label = os.path.join(path_prefix, 'train.csv')
train_no_label = os.path.join(path_prefix, 'train_unlabel.csv')
testing_data = os.path.join(path_prefix, 'test.csv')
w2v_path = os.path.join(path_prefix, 'w2v_all.model')

# Hyperparameters (optimized for memory efficiency and performance)
sen_len = 30          # Sentence length
fix_embedding = False # Allow embedding to be fine-tuned
batch_size = 256      # Batch size (adjust based on GPU memory)
epoch = 8             # Number of epochs
lr = 0.003            # Learning rate
hidden_dim = 128      # Hidden dimension
num_layers = 1        # Number of LSTM layers
dropout = 0.3         # Dropout rate
bidirectional = True  # Use bidirectional LSTM

model_dir = path_prefix

# Load data
print("loading training data ...")
train_x, y = load_training_data(train_with_label)
train_x_no_label = load_training_data(train_no_label)

print("loading testing data ...")
test_x = load_testing_data(testing_data)

# Train embedding model on all text data
print("\n=== Building Vocabulary Embeddings ===")
all_sentences = train_x + train_x_no_label + test_x
w2v_model = SimpleVocabEmbedding(
    all_sentences, 
    vector_size=250, 
    min_count=2
)
w2v_model.save(w2v_path)
print(f"Embeddings saved to {w2v_path}\n")

# Preprocess labeled training data
print("\n=== Preprocessing Labeled Data ===")
preprocess = Preprocess(train_x, sen_len, w2v_path=w2v_path)
embedding = preprocess.make_embedding(load=True)
train_x_tensor = preprocess.sentence_word2idx()
y_tensor = preprocess.labels_to_tensor(y)

# Split into train/val
split_idx = int(len(train_x_tensor) * 0.9)
X_train, X_val = train_x_tensor[:split_idx], train_x_tensor[split_idx:]
y_train, y_val = y_tensor[:split_idx], y_tensor[split_idx:]

print(f"Training samples: {len(X_train)}, Validation samples: {len(X_val)}")

# Create datasets and loaders
train_dataset = SenDataset(X=X_train, y=y_train)
val_dataset = SenDataset(X=X_val, y=y_val)

train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = torch.utils.data.DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

# Initialize model
model = LSTM_Net(
    embedding, 
    embedding_dim=250, 
    hidden_dim=hidden_dim, 
    num_layers=num_layers, 
    dropout=dropout, 
    fix_embedding=fix_embedding,
    bidirectional=bidirectional
)
model = model.to(device)

# Initial training
print("\n=== Initial Training ===")
best_acc = training(batch_size, epoch, lr, model_dir, train_loader, val_loader, model, device)

## Self-Training with Unlabeled Data

In [ ]:
print("\n=== Self-Training with Unlabeled Data ===")
preprocess_unlabeled = Preprocess(train_x_no_label, sen_len, w2v_path=w2v_path)
_ = preprocess_unlabeled.make_embedding(load=True)
unlabeled_x = preprocess_unlabeled.sentence_word2idx()

unlabeled_dataset = SenDataset(X=unlabeled_x, y=None)
unlabeled_loader = torch.utils.data.DataLoader(
    dataset=unlabeled_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

# Generate pseudo-labels
print("Generating pseudo-labels...")
pseudo_data = self_training_iteration(model, unlabeled_loader, device, threshold=0.9)
print(f"Generated {len(pseudo_data)} pseudo-labeled samples")

if len(pseudo_data) > 0:
    # Combine original training data with pseudo-labeled data
    combined_X = torch.cat([X_train, torch.stack([x for x, _ in pseudo_data])], dim=0)
    combined_y = torch.cat([y_train, torch.tensor([y for _, y in pseudo_data])], dim=0)
    
    print(f"Combined training set size: {len(combined_X)}")
    
    # Retrain with combined data
    combined_dataset = SenDataset(X=combined_X, y=combined_y)
    combined_loader = torch.utils.data.DataLoader(
        dataset=combined_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )
    
    # Load best model and continue training
    model = torch.load(os.path.join(model_dir, 'ckpt.model'), weights_only=False)
    model = model.to(device)
    
    # Fine-tune with lower learning rate
    print("\n=== Fine-tuning with Pseudo-labeled Data ===")
    training(batch_size, 5, lr * 0.5, model_dir, combined_loader, val_loader, model, device)

## Final Prediction

In [ ]:
print("\n=== Final Prediction ===")
preprocess_test = Preprocess(test_x, sen_len, w2v_path=w2v_path)
_ = preprocess_test.make_embedding(load=True)
test_x_tensor = preprocess_test.sentence_word2idx()

test_dataset = SenDataset(X=test_x_tensor, y=None)
test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print('load model ...')
model = torch.load(os.path.join(model_dir, 'ckpt.model'), weights_only=False)
outputs = testing(batch_size, test_loader, model, device)

# Save predictions
tmp = pd.DataFrame({"id":[str(i) for i in range(len(test_x))], "labels":outputs})
print("save csv ...")
tmp.to_csv(os.path.join(path_prefix, 'predict.csv'), index=False)
print("Finish Predicting")

## Summary

This notebook implements:
1. **Custom Word Embedding**: No gensim dependency, uses randomized embeddings
2. **Bidirectional LSTM**: Better context understanding
3. **Multi-layer Classifier**: With dropout for regularization
4. **Self-Training**: Utilizes unlabeled data with pseudo-labeling
5. **GPU Acceleration**: Ready for Google Colab GPU

### Key Hyperparameters to Tune:
- `sen_len`: Sentence length (default: 30)
- `batch_size`: Batch size (default: 256)
- `epoch`: Number of epochs (default: 8)
- `lr`: Learning rate (default: 0.003)
- `hidden_dim`: Hidden dimension (default: 128)
- `dropout`: Dropout rate (default: 0.3)
- `bidirectional`: Use bidirectional LSTM (default: True)
- `fix_embedding`: Whether to fine-tune embeddings (default: False)

### Tips for Better Results:
1. Try different learning rates (0.001, 0.002, 0.005)
2. Adjust batch size based on GPU memory
3. Experiment with more epochs if not overfitting
4. Try different dropout rates
5. Analyze validation accuracy trends to adjust hyperparameters

### Output Files:
- `w2v_all.model`: Saved word embeddings
- `ckpt.model`: Best model checkpoint
- `predict.csv`: Final predictions (saved in your 6001-HW4 folder)